In [1]:
import watermark
pkg_versions = watermark.watermark(
    packages="requests,pandas,tqdm")
print(pkg_versions)

requests: 2.32.5
pandas  : 1.5.3
tqdm    : 4.67.1



In [2]:
import os
import requests
from dotenv import load_dotenv
import pandas as pd
from tqdm import tqdm

load_dotenv()
API_KEY = os.getenv("API_KEY")
BASE_URL = "https://apis.data.go.kr"

# 행정안전부_행정표준코드_법정동코드

In [121]:
# https://www.data.go.kr/iim/api/selectAPIAcountView.do
URL = f"{BASE_URL}/1741000/StanReginCd/getStanReginCdList"
params = {
    "serviceKey":API_KEY,
    "numOfRows": 1000,
    "pageNo": 1,
    "flag":"Y",
    "locatadd_nm":"부산광역시",
    "type":"json"
}
res = requests.get(URL, params= params)
data = res.json()
station_df = pd.DataFrame(data["StanReginCd"][1]["row"])
station_df["signguCode"] = (
    station_df["sido_cd"].astype(str).str.zfill(2)
    + station_df["sgg_cd"].astype(str).str.zfill(3)
)
busan_df = station_df[
    station_df["locallow_nm"].str.contains(
        r".*(?:구|군)$", na=False
    )
].reset_index(drop=True)

In [122]:
busan_df

,region_cd,sido_cd,sgg_cd,umd_cd,ri_cd,locatjumin_cd,locatjijuk_cd,locatadd_nm,locat_order,locat_rm,locathigh_cd,locallow_nm,adpt_de,signguCode
0,2611000000,26,110,000,00,2611000000,2611000000,부산광역시 중구,1,,2600000000,중구,,26110
1,2614000000,26,140,000,00,2614000000,2614000000,부산광역시 서구,2,,2600000000,서구,,26140
2,2617000000,26,170,000,00,2617000000,2617000000,부산광역시 동구,3,,2600000000,동구,,26170
3,2620000000,26,200,000,00,2620000000,2620000000,부산광역시 영도구,4,,2600000000,영도구,,26200
4,2623000000,26,230,000,00,2623000000,2623000000,부산광역시 부산진구,5,,2600000000,부산진구,,26230
5,2626000000,26,260,000,00,2626000000,2626000000,부산광역시 동래구,6,,2600000000,동래구,,26260
6,2629000000,26,290,000,00,2629000000,2629000000,부산광역시 남구,7,,2600000000,남구,,26290
7,2632000000,26,320,000,00,2632000000,2632000000,부산광역시 북구,8,,2600000000,북구,,26320
8,2635000000,26,350,000,00,2635000000,2635000000,부산광역시 해운대구,9,,2600000000,해운대구,,26350
9,2638000000,26,380,000,00,2638000000,2638000000,부산광역시 사하구,10,,2600000000,사하구,,26380


In [128]:
busan_codes = busan_df["signguCode"].unique()

# 한국관광공사_빅데이터_지역별 방문자수_GW

In [123]:
# https://www.data.go.kr/iim/api/selectAPIAcountView.do
URL = f"{BASE_URL}/B551011/DataLabService/locgoRegnVisitrDDList"

In [124]:
params = {
    "serviceKey":API_KEY,
    "numOfRows": 1,
    "pageNo": 1,
    "MobileOS":"IOS",
    "MobileApp":"AppTest",
    "startYmd":"20100101",
    "endYmd":"20260915",
    "_type":"json"
}

In [125]:
res = requests.get(URL, params= params)
data = res.json()
total_count = data["response"]["body"]["totalCount"]

In [131]:
num_of_rows = 10000
params.update({"numOfRows":num_of_rows})

dfs = list()
for page in tqdm(range(1, total_count//num_of_rows + 2)):
    params.update({"pageNo":page})
    res = requests.get(URL, params= params)
    data = res.json()
    df = pd.DataFrame(data["response"]["body"]["items"]["item"])
    dfs.append(df[df["signguCode"].isin(busan_codes)])

100%|██████████| 237/237 [26:42<00:00,  6.76s/it]


In [132]:
df["signguNm"].value_counts()

동구      180
중구      177
서구      144
남구      144
북구      144
       ... 
동대문구     33
광진구      33
성동구      33
용산구      33
순천시      33
Name: signguNm, Length: 250, dtype: int64

In [133]:
df = pd.concat(dfs,ignore_index=True)

In [135]:
df.to_csv("한국관광공사_빅데이터_지역별 방문자수_GW.csv", index=False)